In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly as py 
import plotly.tools as tls 
import plotly.express as px

In [ ]:
marks= pd.read_excel("C:/Data_Science/Pandas/Student Performance Analysis.xlsx",sheet_name='Marks')


In [ ]:
attendance= pd.read_excel("C:/Data_Science/Pandas/Student Performance Analysis.xlsx",sheet_name='Attendance')

In [ ]:
##########column transformation
marks['Name']= marks['Name'].str.strip().str.title()
attendance['Name']= attendance['Name'].str.strip().str.title()

In [ ]:
##### data cleansing

marks=marks.replace(' ',0)
marks=marks.fillna(0)


In [ ]:

attendance.replace('Y',1,inplace=True)
attendance.replace('N',0,inplace=True)

In [ ]:
marks['Mini Test 1']=marks['Mini Test 1'].astype('int64')
marks['Mini Test 2']=marks['Mini Test 2'].astype('int64')
marks['Live Test']=marks['Live Test'].astype('int64')
marks['Assignment']=marks['Assignment'].astype('int64')
marks.replace(113,6,inplace=True)

In [ ]:
#merge both sheets
stud_performance= pd.merge(marks,attendance,on='Name',how='left')
#creating column Total Marks
stud_performance['Total Marks'] = stud_performance[['Mini Test 1','Mini Test 2','Live Test','Assignment']].sum(axis=1) #summing horizontally
stud_performance.groupby('Name').sum()

#calcuting maximum marks
mini_test1 = max(stud_performance['Mini Test 1'])
mini_test2 = max(stud_performance['Mini Test 2'])
live_test = max(stud_performance['Live Test'])
assignment_m = max(stud_performance['Assignment'])
total_marks = mini_test1+mini_test2+live_test+assignment_m

#Percentage
stud_performance['Percentage'] = round((stud_performance['Total Marks']/total_marks*100),1)
stud_performance['Total Attendance']=stud_performance[['Attendance Day 1','Attendance Day 2','Attendance Day 3','Attendance Day 4','Attendance Day 5']].sum(axis=1)
stud_performance['Attendance Percentage'] = stud_performance[['Attendance Day 1','Attendance Day 2','Attendance Day 3','Attendance Day 4','Attendance Day 5']].mean(axis=1) * 100

#calculating marks percent for each test
stud_performance['per_mini_test1'] = stud_performance['Mini Test 1']/mini_test1
stud_performance['per_mini_test2'] = stud_performance['Mini Test 2']/mini_test2
stud_performance['per_live_test'] = stud_performance['Live Test']/live_test
stud_performance['per_assignment'] = stud_performance['Assignment']/assignment_m
stud_performance['attendance_per']= stud_performance['Total Attendance']/5

In [ ]:
#calculating weighted percentage
stud_performance['Weighted Percentage'] = round((stud_performance['attendance_per'] * 0.4
                                                 +stud_performance['per_mini_test1'] * 0.1 
                                                 +stud_performance['per_mini_test2'] * 0.1 
                                                 +stud_performance['per_live_test'] * 0.2 
                                                 +stud_performance['per_assignment'] * 0.2)*100)

#creating performance category
def category(df):
    if df['Percentage']>=85:
        return "Excellent" 
    elif df['Percentage']>=71 and df['Weighted Percentage']<=84:
        return "Good"
    elif df['Weighted Percentage']>=50 and df['Weighted Percentage']<=70:
        return "Average"
    else:
        return "Needs Improvement" 

In [ ]:
#adding category to each element
stud_performance['Performance'] = stud_performance.apply(category,axis=1)

In [ ]:
d = stud_performance[(stud_performance['Attendance Percentage'] < 75) & (stud_performance['Weighted Percentage'] > 50)]
print('Students with above criteria : ',d[['Name','Attendance Percentage','Weighted Percentage']])

In [ ]:

top = stud_performance.nlargest(3,columns='Percentage')
print(top[['Name','Percentage','Total Marks','Performance','Attendance Percentage']])

In [ ]:

stud_performance[['Mini Test 1','Mini Test 2','Live Test','Assignment','Total Attendance']].corr()

In [ ]:
#1.	Create a bar chart displaying weighted percentages for top 5 students.

top5 = stud_performance.nlargest(5,columns='Weighted Percentage')
print(top5[['Name','Weighted Percentage']])

fig = px.bar(top5, x="Name", y="Weighted Percentage", barmode="group",title='top 5 students')
fig.show()

In [ ]:
#2.	Create a pie chart showing the distribution of students across the four performance categories.

In [ ]:
perf = pd.DataFrame(stud_performance['Performance'].value_counts()).reset_index()
perf.columns=['Performance','Count']

In [ ]:
perf

In [ ]:
# pie chart
fig = px.pie(perf, values='Count', names='Performance', title='Dist Students Across Performance Categories')
fig.show()

In [ ]:
#3.	Create box plots for each test (Live Test, Mini Test 1, Mini Test 2, Assignment) to visualize the spread and detect potential outliers in scores.

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle('Box Plots for Each Test', fontsize=20)

sns.boxplot(y=stud_performance['Mini Test 1'], ax=axes[0, 0])
axes[0, 0].set_title('Mini Test 1')
axes[0, 0].set_ylabel('Marks')

sns.boxplot(y=stud_performance['Mini Test 2'], ax=axes[0, 1])
axes[0, 1].set_title('Mini Test 2')
axes[0, 1].set_ylabel('Marks')

sns.boxplot(y=stud_performance['Live Test'], ax=axes[1, 0])
axes[1, 0].set_title('Live Test')
axes[1, 0].set_ylabel('Marks')

sns.boxplot(y=stud_performance['Assignment'], ax=axes[1, 1])
axes[1, 1].set_title('Assignment')
axes[1, 1].set_ylabel('Marks')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
#4.	Create a chart to show the students where attendance is less than 50%.

less_than_50 = stud_performance[stud_performance['Attendance Percentage']<50]
print(less_than_50[['Name','Attendance Percentage']])

fig = px.bar(less_than_50, x="Name", y="Attendance Percentage", barmode="group",title='Attendance less than 50%')
fig.show()